In [1]:
# RUN THIS CELL FIRST TO IMPORT ALL NECESSARY LIBRARIES
import pandas as pd
import numpy as np
import requests
import time
import io
import os

### EXTRACT DATA

In [ ]:
# RUN THIS CELL ONLY TO UPDATE THE ISTAT DATA
# this cell save the istat data in a raw data folder in order to avoid running the api call every time
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
header = {"Accept":"application/vnd.sdmx.data+csv;version=1.0.0"}
response = requests.get(url, headers=header)
print(response.status_code)

with open("data/raw/istat_incidenti.csv", "w", encoding="utf-8") as f:
    f.write(response.text)

200


In [30]:
# RUN THIS CELL ONCE TO DOWNLOAD SITUAS DATA
# this cell save the situas data from 31-12-2001 to 31-12-2024 in 24 different csv files
date_ranges = [f"{year}-12-31" for year in range(2001, 2025)]

# to download files in csv format is necessary to set json and header parameters.
# You can acknowledge this by using dev tools on the web page at the below specified url.
json_situas = {"orderFields":[],"orderDirects":[],"pFilterFields":[],"pFilterValues":[]}
header_situas = {"Content-Type": "application/json-patch+json"}

for i in date_ranges:
    url_situas = f"https://situas.istat.it/ShibO2Module/api/Report/Spool/{i}/74?&pdoctype=CSV"
    time.sleep(1)
    response_situas = requests.post(url_situas, json = json_situas, headers = header_situas)
    print(response_situas.status_code)
    if response_situas.status_code == 200:
        with open(f"data/raw/situas_{i[0:4]}.csv", "w", encoding="utf-8") as f:
            f.write(response_situas.text)
    else:
        # if an http status code different from 200 is encounterd, this make sure that cell execution is stopped
        raise SystemExit(f"Abort due to http error {response_situas.status_code}")
print(f"All done - {len(date_ranges)} files downloaded")


200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
All done - 24 files downloaded


### CLEAN DATA

In [2]:
situas_df = pd.read_csv("data/raw/situas_2001.csv", sep = ";", decimal = ",")
situas_df["Anno"] = 2001
print(f"Year 2001:{situas_df.shape}")
situas_df = [situas_df]

for i in range(2002,2025):
    tmp_df = pd.read_csv(f"data/raw/situas_{i}.csv", sep = ";", decimal = ",")
    tmp_df["Anno"] = i # add a column with the year of the data

# Analyzing the csv situas data, it can be noticed that from 2015 on there is another column 
# not present in the previous years "Codice Provincia (Storico)"". This is simply the code that 
# in the previous years is called "Codice Provincia/Uts" which from 2015 on contains different values. 
# So, we can drop "Codice Provincia/Uts" and change "Codice Provincia (Storico)"" name in "Codice Provincia/Uts"
# in order to obtain a uniform dataframe.
    if i in range(2015,2025):
        tmp_df.drop(columns=["Codice Provincia/Uts"], inplace = True)
        tmp_df.rename(columns={"Codice Provincia (Storico)": "Codice Provincia/Uts"}, inplace = True)

    # check if columns name are the same among csv files in both directions
    check_column_names_sx = set(situas_df[0].columns) - set(tmp_df.columns)
    check_column_names_dx = set(tmp_df.columns) - set(situas_df[0].columns)

    # check if columns order are the same among csv files
    check_column_order = list(situas_df[0].columns) == list(tmp_df.columns)

    if len(check_column_names_sx) == 0 and len(check_column_names_dx) == 0 and check_column_order:
        print(f"Year {i}: {tmp_df.shape}")
        situas_df.append(tmp_df)
    else:
        print(situas_df[0].columns)
        print(tmp_df.columns)
        raise SystemExit(f"Abort due to column name mismatch in situas_{i}.csv")
situas_df = pd.concat(situas_df, ignore_index = True)



Year 2001:(8102, 17)
Year 2002: (8102, 17)
Year 2003: (8100, 17)
Year 2004: (8101, 17)
Year 2005: (8101, 17)
Year 2006: (8101, 17)
Year 2007: (8101, 17)
Year 2008: (8101, 17)
Year 2009: (8100, 17)
Year 2010: (8094, 17)
Year 2011: (8092, 17)
Year 2012: (8092, 17)
Year 2013: (8090, 17)
Year 2014: (8057, 17)
Year 2015: (8046, 17)
Year 2016: (7998, 17)
Year 2017: (7978, 17)
Year 2018: (7954, 17)
Year 2019: (7914, 17)
Year 2020: (7903, 17)
Year 2021: (7904, 17)
Year 2022: (7904, 17)
Year 2023: (7900, 17)
Year 2024: (7896, 17)


From the shapes of the previous cell, we can notice that the number of rows change for every situas csv file. Since that every record in these csv files is a different town, this means that the number of town changes every year. This happens because some towns splits or merge toghether with others. This kind of town usually are very small so it's difficult that they could be interesting for investment from business. So we can simply keep just the towns that are present in every csv file. In order to confirm this strategy, we can calculate how many towns are in common between all csv files.

In [7]:
# how many years are in the dataset (24)
tot_years = situas_df["Anno"].nunique()

# for every town, count in how many years it appears.
count_years_per_town = situas_df.groupby("Codice Comune (alfanumerico)")["Anno"].nunique()

# keep only the towns that appear in all years
towns_common = count_years_per_town[count_years_per_town == tot_years].index
print(len(towns_common))

7488


Since that the majority of towns don't change through years, we can assess that the bigger towns are not excluded from the analysis and we can confirm the strategy.

In [11]:
print(situas_df.info())
situas_df.sample(10)

<class 'pandas.DataFrame'>
RangeIndex: 192731 entries, 0 to 192730
Data columns (total 17 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   ï»¿Codice Ripartizione geografica  192731 non-null  int64  
 1   Codice Regione                     192731 non-null  int64  
 2   Codice Provincia/Uts               192731 non-null  int64  
 3   Codice Comune (alfanumerico)       192731 non-null  int64  
 4   Codice Comune (numerico)           192731 non-null  int64  
 5   Comune                             192707 non-null  str    
 6   Comune (dizione straniera)         2911 non-null    str    
 7   Sigla automobilistica              190523 non-null  str    
 8   Capoluogo di Provincia/Uts         192731 non-null  int64  
 9   Capoluogo di Regione               192731 non-null  int64  
 10  Popolazione legale                 192722 non-null  float64
 11  Anno Censimento                    192731 non-null

,ï»¿Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),Anno
84145,2,5,24,24018,24018,Caldogno,NaN,VI,0,0,11221.0,2011,"15,8845",2011,11270,2011,2011
67674,2,4,22,22057,22057,Cimego,NaN,TN,0,0,407.0,2001,"10,5139",2009,406,2009,2009
87191,4,16,75,75053,75053,Neviano,NaN,LE,0,0,5514.0,2011,"16,3019",2011,5517,2011,2011
118506,4,15,62,62065,62065,San Martino Sannita,NaN,BN,0,0,1277.0,2011,"6,1840",2015,1208,2015,2015
99666,1,3,18,18161,18161,Torricella Verzate,NaN,PV,0,0,837.0,2011,"3,6271",2013,821,2013,2013
57409,1,1,4,4213,4213,Santo Stefano Belbo,NaN,CN,0,0,4037.0,2001,"23,5721",2008,4106,2008,2008
64895,1,1,1,1087,1087,Claviere,NaN,TO,0,0,163.0,2001,"2,6872",2009,197,2009,2009
62264,4,15,65,65008,65008,Aquara,NaN,SA,0,0,1799.0,2001,"32,7339",2008,1668,2008,2008
116392,2,5,23,23052,23052,Negrar,NaN,VR,0,0,16935.0,2011,"40,4219",2015,17051,2015,2015
166454,4,15,65,65054,65054,Futani,NaN,SA,0,0,1100.0,2021,"14,8506",2021,1100,2021,2021


In [ ]:
# drop the column "Comune (dizione straniera) since that have a lot nan values and is not useful for our analysis"
situas_df.drop(columns=['Comune (dizione straniera)'],inplace=True)

In [ ]:
# Loading the situas data, it can be noticed that the first column name is not correct. 
# It is called "ï»¿Codice Ripartizione geografica" instead of "Codice Ripartizione geografica". 
# This is due to a problem with the encoding of the csv file. We can fix this by renaming the column.
situas_df.rename(columns={'ï»¿Codice Ripartizione geografica': 'Codice Ripartizione geografica'}, inplace=True)
print(situas_df.info())
print(situas_df.sample(10))

<class 'pandas.DataFrame'>
RangeIndex: 192731 entries, 0 to 192730
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Codice Ripartizione geografica  192731 non-null  int64  
 1   Codice Regione                  192731 non-null  int64  
 2   Codice Provincia/Uts            192731 non-null  int64  
 3   Codice Comune (alfanumerico)    192731 non-null  int64  
 4   Codice Comune (numerico)        192731 non-null  int64  
 5   Comune                          192707 non-null  str    
 6   Sigla automobilistica           190523 non-null  str    
 7   Capoluogo di Provincia/Uts      192731 non-null  int64  
 8   Capoluogo di Regione            192731 non-null  int64  
 9   Popolazione legale              192722 non-null  float64
 10  Anno Censimento                 192731 non-null  int64  
 11  Superficie (Kmq)                192731 non-null  str    
 12  Anno (Superficie)          

From the previous cell we can see there are some nan values (about 2000) in the column "Sigla automobilistica".

In [26]:
print(situas_df[pd.isna(situas_df['Sigla automobilistica'])])
print(situas_df['Codice Regione'][pd.isna(situas_df['Sigla automobilistica'])].unique())

        Codice Ripartizione geografica  Codice Regione  Codice Provincia/Uts  \
5339                                 4              15                    63   
5340                                 4              15                    63   
5341                                 4              15                    63   
5342                                 4              15                    63   
5343                                 4              15                    63   
...                                ...             ...                   ...   
189978                               4              15                    63   
189979                               4              15                    63   
189980                               4              15                    63   
189981                               4              15                    63   
189982                               4              15                    63   

        Codice Comune (alfanumerico)  C

Investigating on this columns, we can see that all nan values have "Codice Regione" $= 15$. This code stands for "Campania". We can fill these nan with the correct code using the column "Codice Provincia"

In [31]:
# we can discover the province code for each of the 5 province of Campania.
# Then we can fill nan values in column "Sigla automobilistica" with the correct string code.
province_code = []
province_name = []
for i in range(0,len(situas_df)):
    if situas_df['Comune'].iloc[i] in ['Napoli','Caserta','Salerno','Avellino','Benevento']:
        province_code.append(situas_df['Codice Provincia/Uts'].iloc[i])
        province_name.append(situas_df['Comune'].iloc[i])
province_campania_df = pd.DataFrame({'Codice Provincia/Uts': province_code, 'Nome Provincia/Uts': province_name})
print(province_campania_df)

for i in range(0,len(situas_df)):
    if pd.isna(situas_df['Sigla automobilistica'].iloc[i]):
        if situas_df['Codice Provincia/Uts'].iloc[i] == 61:
            situas_df.loc[i,'Sigla automobilistica'] = 'CE'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 62:
            situas_df.loc[i,'Sigla automobilistica'] = 'BN'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 63:
            situas_df.loc[i,'Sigla automobilistica']  = 'NA'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 64:
            situas_df.loc[i,'Sigla automobilistica'] = 'AV'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 65:
            situas_df.loc[i,'Sigla automobilistica'] = 'SA'

print(situas_df.info())
print(situas_df.sample(10))


     Codice Provincia/Uts Nome Provincia/Uts
0                      61            Caserta
1                      62          Benevento
2                      63             Napoli
3                      64           Avellino
4                      65            Salerno
..                    ...                ...
115                    61            Caserta
116                    62          Benevento
117                    63             Napoli
118                    64           Avellino
119                    65            Salerno

[120 rows x 2 columns]
<class 'pandas.DataFrame'>
RangeIndex: 192731 entries, 0 to 192730
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Codice Ripartizione geografica  192731 non-null  int64  
 1   Codice Regione                  192731 non-null  int64  
 2   Codice Provincia/Uts            192731 non-null  int64  
 3   Codice Comune (alfanu

In [32]:
situas_df.to_csv("data/clean/situas_df.csv", index=False, sep = ";", decimal = ",")

Now, we can move to clean ISTAT data csv file